In [1]:
import os
import sys
import shutil
import time
import random
import argparse
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torchvision.datasets as dset
import torchvision.transforms as transforms
from utils_.utils import AverageMeter, RecorderMeter, time_string, convert_secs2time
from tensorboardX import SummaryWriter
import models
import copy
import numpy as np
from models.attack_model import Attack
from models.nomarlization_layer import Normalize_layer, noise_Normalize_layer
from numpy.random import RandomState
import copy
import csv
from scipy import stats

In [2]:
from argparse import Namespace
# 创建一个命名空间对象
args = Namespace()
args.dataset = 'cifar10'
args.adv_eval = False
args.data_path = '/home/xueqiong/CVPR_2019_PNI-master/data'
args.batch_size = 500
args.workers = 4
args.input_noise = False
args.learning_rate = 0.1
args.momentum = 0.9
args.decay = 0.0003
args.fine_tune = False
# -------------上面的基本上不用动，下面的要改
args.use_cuda = True
args.ngpu = 0
args.folder = "cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005"
if 'fixnoise_resnet20' in args.folder:
    args.arch = 'fixnoise_resnet20'
    args.fix_noise_level = float(args.folder.split('_')[-1])
    print('fix_noise_level: ', args.fix_noise_level)
elif 'fixnoise_resnet8' in args.folder:
    args.arch = 'fixnoise_resnet8'
    args.fix_noise_level = float(args.folder.split('_')[-1])
    print('fix_noise_level: ', args.fix_noise_level)
elif 'noise_resnet20' in args.folder:
    args.arch = 'noise_resnet20'
elif 'noise_resnet8' in args.folder:
    args.arch = 'noise_resnet8'
elif 'dropout_resnet20' in args.folder:
    args.arch = 'dropout_resnet20'
elif 'dropout_resnet8' in args.folder:
    args.arch = 'dropout_resnet8'
elif 'vanilla_resnet20' in args.folder:
    args.arch = 'vanilla_resnet20'
elif 'vanilla_resnet8' in args.folder:
    args.arch = 'vanilla_resnet8'
    
args.num_filters = 4

if '_adv' in args.folder:
    args.adv_train = True
else:
    args.adv_train = False
# args.resume_path = '/home/xueqiong/noise_injection_uncertainty/cifar10/save/final_res_resnet8/{}'.format(args.folder)
args.resume_path = '/home/xueqiong/noise_injection_uncertainty/cifar10/save/2025-04-23/cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005'
args.save_path = os.path.join(args.resume_path, 'test')
args.manualSeed = 123
args.optimizer = 'SGD'

fix_noise_level:  0.005


In [3]:
def print_log(print_string, log):
    print("{}".format(print_string))
    log.write('{}\n'.format(print_string))
    log.flush()

In [4]:
# Init logger
if not os.path.isdir(args.save_path):
    os.makedirs(args.save_path)
log = open(os.path.join(args.save_path,
                        'log_seed_{}.txt'.format(args.manualSeed)), 'w')
if args.use_cuda == False:
    print_log('No GPU!', log)
    print_log('{},{}'.format(args.ngpu,torch.cuda.is_available()), log)
print_log('save path : {}'.format(args.save_path), log)
state = {k: v for k, v in args._get_kwargs()}
print_log(state, log)
print_log("Random Seed: {}".format(args.manualSeed), log)
print_log("python version : {}".format(
    sys.version.replace('\n', ' ')), log)
print_log("torch  version : {}".format(torch.__version__), log)
print_log("cudnn  version : {}".format(
    torch.backends.cudnn.version()), log)

# Init the tensorboard path and writer
tb_path = os.path.join(args.save_path, 'tb_log')
writer = SummaryWriter(tb_path)

save path : /home/xueqiong/noise_injection_uncertainty/cifar10/save/2025-04-23/cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005/test
{'dataset': 'cifar10', 'adv_eval': False, 'data_path': '/home/xueqiong/CVPR_2019_PNI-master/data', 'batch_size': 500, 'workers': 4, 'input_noise': False, 'learning_rate': 0.1, 'momentum': 0.9, 'decay': 0.0003, 'fine_tune': False, 'use_cuda': True, 'ngpu': 0, 'folder': 'cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005', 'arch': 'fixnoise_resnet8', 'fix_noise_level': 0.005, 'num_filters': 4, 'adv_train': False, 'resume_path': '/home/xueqiong/noise_injection_uncertainty/cifar10/save/2025-04-23/cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005', 'save_path': '/home/xueqiong/noise_injection_uncertainty/cifar10/save/2025-04-23/cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005/test', 'manualSeed': 123, 'optimizer': 'SGD'}
Random Seed: 123
python version : 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:12:24) [GCC 11.2.0]
torch  version : 2.4.1+cu124
cudn

In [5]:
if args.dataset == 'cifar10':
    mean = [x / 255 for x in [125.3, 123.0, 113.9]]
    std = [x / 255 for x in [63.0, 62.1, 66.7]]
elif args.dataset == 'cifar100':
    mean = [x / 255 for x in [129.3, 124.1, 112.4]]
    std = [x / 255 for x in [68.2, 65.4, 70.4]]
elif args.dataset == 'svhn':
    mean = [0.5, 0.5, 0.5]
    std = [0.5, 0.5, 0.5]
elif args.dataset == 'mnist':
    mean = [0.5, 0.5, 0.5]
    std = [0.5, 0.5, 0.5]
elif args.dataset == 'imagenet':
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
else:
    assert False, "Unknow dataset : {}".format(args.dataset)

# Current data-preprocessing does not include the normalization
imagenet_train_transform = [
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()]
imagenet_test_transform = [
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()]

normal_train_transform = [
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor()]
normal_test_transform = [
    transforms.ToTensor()]

# if not performing the adversarial training or evalutaion, we append
# the normalization back to the preprocessing
if not (args.adv_train or args.adv_eval):
    imagenet_train_transform.append(transforms.Normalize(mean, std))
    imagenet_test_transform.append(transforms.Normalize(mean, std))
    normal_train_transform.append(transforms.Normalize(mean, std))
    normal_test_transform.append(transforms.Normalize(mean, std))

if args.dataset == 'imagenet':
    train_transform = transforms.Compose(imagenet_train_transform)
    test_transform = transforms.Compose(imagenet_test_transform)
else:
    train_transform = transforms.Compose(normal_train_transform)
    test_transform = transforms.Compose(normal_test_transform)

if args.dataset == 'mnist':
    train_data = dset.MNIST(
        args.data_path, train=True, transform=train_transform, download=True)
    test_data = dset.MNIST(args.data_path, train=False,
                            transform=test_transform, download=True)
    num_classes = 10
elif args.dataset == 'cifar10':
    train_data = dset.CIFAR10(
        args.data_path, train=True, transform=train_transform, download=True)
    test_data = dset.CIFAR10(
        args.data_path, train=False, transform=test_transform, download=True)
    num_classes = 10
elif args.dataset == 'cifar100':
    train_data = dset.CIFAR100(
        args.data_path, train=True, transform=train_transform, download=True)
    test_data = dset.CIFAR100(
        args.data_path, train=False, transform=test_transform, download=True)
    num_classes = 100
elif args.dataset == 'svhn':
    train_data = dset.SVHN(args.data_path, split='train',
                            transform=train_transform, download=True)
    test_data = dset.SVHN(args.data_path, split='test',
                            transform=test_transform, download=True)
    num_classes = 10
elif args.dataset == 'stl10':
    train_data = dset.STL10(
        args.data_path, split='train', transform=train_transform, download=True)
    test_data = dset.STL10(args.data_path, split='test',
                            transform=test_transform, download=True)
    num_classes = 10
elif args.dataset == 'imagenet':
    train_dir = os.path.join(args.data_path, 'train')
    test_dir = os.path.join(args.data_path, 'val')
    train_data = dset.ImageFolder(train_dir, transform=train_transform)
    test_data = dset.ImageFolder(test_dir, transform=test_transform)
    num_classes = 1000
else:
    assert False, 'Do not support dataset : {}'.format(args.dataset)

test_loader = torch.utils.data.DataLoader(test_data, batch_size=args.batch_size, shuffle=False,
                                            num_workers=args.workers, pin_memory=False)

Files already downloaded and verified
Files already downloaded and verified


In [6]:
# Init model, criterion, and optimizer
if 'fixnoise' not in args.arch:
    net_c = models.__dict__[args.arch](num_classes, args.num_filters)
else:
    net_c = models.__dict__[args.arch](num_classes, args.num_filters, args.fix_noise_level)
        # For adversarial case, override the original network with normalization layer
if (args.adv_train or args.adv_eval):
    if not args.input_noise:
        net = torch.nn.Sequential(
                Normalize_layer(mean,std),
                net_c
                )
    else:
        net = torch.nn.Sequential(
                noise_Normalize_layer(mean,std),
                net_c
                )           
else:
    net = net_c

CifarResNet : Depth : 8 , Layers for each block : 1


/home/xueqiong/noise_injection_uncertainty/cifar10/models/fixnoisy_resnet_cifar.py:91: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  init.kaiming_normal(m.weight)


In [7]:
print_log("=> network :\n {}".format(net), log)

if args.use_cuda:
    if args.ngpu > 1:
        net = torch.nn.DataParallel(net, device_ids=list(range(args.ngpu)))

# define loss function (criterion) and optimizer
criterion = torch.nn.CrossEntropyLoss()

if args.use_cuda:
    net.cuda()
    criterion.cuda()

=> network :
 CifarResNet(
  (conv_1_3x3): fixnoise_Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn_1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (stage_1): Sequential(
    (0): ResNetBasicblock(
      (conv_a): fixnoise_Conv2d(4, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn_a): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv_b): fixnoise_Conv2d(4, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn_b): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (stage_2): Sequential(
    (0): ResNetBasicblock(
      (conv_a): fixnoise_Conv2d(4, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn_a): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv_b): fixnoise_Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)

In [8]:
def recursive_module_iteration(module, depth=0):
    # 打印当前模块的信息
    print(f"{' ' * (depth * 2)}Module: {module.__class__.__name__}")

    # 如果当前模块是容器模块（如 Sequential），则递归遍历其子模块
    if hasattr(module, "children"):
        for child in module.children():
            recursive_module_iteration(child, depth + 1)
    if isinstance(module, nn.Dropout):
        module.train()
        print('dropout设置为训练模式')


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].contiguous().view(-1).float().sum(0)
            res.append(correct_k.mul_(100.0 / batch_size))

        return res
    
def validate(val_loader, model, criterion, log, test_times=10, return_detail=False):
    losses = AverageMeter()
    acc_avg = AverageMeter()
    bs = AverageMeter()
    # switch to evaluate mode
    samplesize = val_loader.dataset.data.shape[0]
    model.eval()
    if 'dropout' in args.arch:
        recursive_module_iteration(model)
    with torch.no_grad():
        var_logits_all = []
        pred_all = []
        confidence_all  = []
        target_all = []
        avg_probs_all = []
        for i, (input, target) in enumerate(val_loader):
            logits = torch.zeros((input.shape[0], num_classes, test_times))
            if args.use_cuda:
                target = target.cuda(non_blocking=True)
                input = input.cuda()
                logits = logits.cuda()
            for i in range(test_times):
                # compute output
                output = model(input)
                logits[:, :, i] = output
            avg_logits = torch.mean(logits, dim=2)
            avg_probs = torch.nn.functional.softmax(avg_logits, dim=1)
            var_logits = torch.sum(torch.var(logits, dim=2), dim=-1)   # [batch_size]
            if args.use_cuda:
                var_logits = var_logits.cpu()
            var_logits = var_logits.numpy().tolist()
            loss = criterion(avg_logits, target)
            # measure accuracy and record loss
            prec1, prec5 = accuracy(avg_logits.data, target, topk=(1, 5))
            acc_avg.update(prec1.item(), input.size(0))
            bs_ = brier_score(avg_probs.cpu().numpy(), target.cpu().numpy())
            bs.update(bs_, input.size(0))
            _, pred = output.topk(1, 1, True, True)
            if args.use_cuda:
                pred = pred.cpu()
            pred = pred.t()[0].numpy().tolist()
            if return_detail:
                var_logits_all.extend(var_logits)
                pred_all.extend(pred)
                confidence, prediction = torch.max(avg_probs, 1)
                confidence_all.extend(confidence.cpu().numpy().tolist())
                target_all.extend(target.cpu().numpy().tolist())
                avg_probs_all.extend(avg_probs.cpu().numpy().tolist())
    print_log(
    '  **Test** Prec@1 {top1.avg:.3f}'.format(top1=acc_avg), log)
    return acc_avg, var_logits_all, pred_all, confidence_all, target_all, bs.avg, avg_probs_all

# 计算ECE
def calculate_ece(confidences, predictions, targets, num_bins=10):
    ece = 0.0
    acc = 0.0
    for i in range(num_bins):
        # 计算在每个置信度区间的准确性和置信度的平均值
        mask = (confidences >= i / num_bins) & (confidences < (i + 1) / num_bins)
        if mask.sum() > 0:
            acc = (predictions == targets)[mask].mean()
            avg_confidence = confidences[mask].mean()
            ece += abs(acc - avg_confidence) * mask.sum() / len(confidences)
    return ece

# 假设confidences是模型的预测概率值（每个类别的概率），而labels是真实的类别标签
def brier_score(confidences, labels):
    # 将真实标签转换成one-hot编码
    one_hot_labels = np.eye(confidences.shape[1])[labels]
    
    # 计算Brier分数
    brier_score = np.mean(np.sum((confidences - one_hot_labels) ** 2, axis=1))
    
    return brier_score


# predictive performance

In [9]:
acc = []
eces = []
brier_scores = []
num = 0
for package in os.listdir(args.resume_path):
    if package == 'test' or '.npy' in package:
        continue
    resume = os.path.join(args.resume_path, package, 'checkpoint_epoch160.pth.tar')
    print_log("=> loading checkpoint '{}'".format(resume), log)
    if args.use_cuda == True:
        checkpoint = torch.load(resume)
    else:
        checkpoint = torch.load(resume, map_location='cpu')
    if not (args.fine_tune):
        args.start_epoch = checkpoint['epoch']
        recorder = checkpoint['recorder']

    state_tmp = net.state_dict()
    if 'state_dict' in checkpoint.keys():
        state_tmp.update(checkpoint['state_dict'])
    else:
        state_tmp.update(checkpoint)

    net.load_state_dict(state_tmp)

    print_log("=> loaded checkpoint '{}' (epoch {})".format(
        resume, args.start_epoch), log)
    cur_acc, var_logits_all, pred_all, confidence_all, target_all, bs, avg_probs_all = validate(test_loader, net, criterion, log, 100, return_detail=True)
    ece = calculate_ece(np.asarray(confidence_all), np.asarray(pred_all), np.asarray(target_all), num_bins=10)
    acc.append(copy.deepcopy(cur_acc))
    eces.append(ece)
    brier_scores.append(bs)
    num += 1

    # v6
    from sklearn.metrics import roc_auc_score

    true_or_false = np.asarray(pred_all)==np.asarray(target_all)
    print(roc_auc_score(true_or_false[:10000], confidence_all))
    if num == 5:
        break
    
if num != 5:
    print('num!=5, num={}'.format(num))

=> loading checkpoint '/home/xueqiong/noise_injection_uncertainty/cifar10/save/2025-04-23/cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005/1895/checkpoint_epoch160.pth.tar'
=> loaded checkpoint '/home/xueqiong/noise_injection_uncertainty/cifar10/save/2025-04-23/cifar10_fixnoise_resnet8_4_160_SGD_notadv_0.005/1895/checkpoint_epoch160.pth.tar' (epoch 160)


/tmp/ipykernel_468190/4002384515.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume)


  **Test** Prec@1 72.910
0.8133823210950126
num!=5, num=1


In [10]:
if 'fixnoise' in args.arch:
    print([net.state_dict()[name] for name in checkpoint['state_dict'].keys() if 'fix' in name])

[tensor([0.0050], device='cuda:0'), tensor([0.0050], device='cuda:0'), tensor([0.0050], device='cuda:0'), tensor([0.0050], device='cuda:0'), tensor([0.0050], device='cuda:0'), tensor([0.0050], device='cuda:0'), tensor([0.0050], device='cuda:0'), tensor([0.0050], device='cuda:0')]


In [11]:
acc_res = [a.avg for a in acc]
np.mean(acc_res), np.std(acc_res)

(72.91000137329101, 0.0)

In [12]:
np.mean(eces), np.std(eces)

(0.013855790431797497, 0.0)

In [13]:
np.mean(brier_scores), np.std(brier_scores)

(0.37666703560373355, 0.0)

In [14]:
import pathlib
def load_new_test_data(version_string='', load_tinyimage_indices=False):
    data_path = '/home/xueqiong/noise_injection_uncertainty/cifar10/CIFAR-10.1-master/datasets'
    filename = 'cifar10.1'
    if version_string == '':
        version_string = 'v7'
    if version_string in ['v4', 'v6', 'v7']:
        filename += '_' + version_string
    else:
        raise ValueError('Unknown dataset version "{}".'.format(version_string))
    label_filename = filename + '_labels.npy'
    imagedata_filename = filename + '_data.npy'
    label_filepath = os.path.abspath(os.path.join(data_path, label_filename))
    imagedata_filepath = os.path.abspath(os.path.join(data_path, imagedata_filename))
    print('Loading labels from file {}'.format(label_filepath))
    assert pathlib.Path(label_filepath).is_file()
    labels = np.load(label_filepath)
    print('Loading image data from file {}'.format(imagedata_filepath))
    assert pathlib.Path(imagedata_filepath).is_file()
    imagedata = np.load(imagedata_filepath)
    assert len(labels.shape) == 1
    assert len(imagedata.shape) == 4
    assert labels.shape[0] == imagedata.shape[0]
    assert imagedata.shape[1] == 32
    assert imagedata.shape[2] == 32
    assert imagedata.shape[3] == 3
    if version_string == 'v6' or version_string == 'v7':
        assert labels.shape[0] == 2000
    elif version_string == 'v4':
        assert labels.shape[0] == 2021

    if not load_tinyimage_indices:
        return imagedata, labels
    else:
        ti_indices_data_path = os.path.join(os.path.dirname(__file__), '../other_data/')
        ti_indices_filename = 'cifar10.1_' + version_string + '_ti_indices.json'
        ti_indices_filepath = os.path.abspath(os.path.join(ti_indices_data_path, ti_indices_filename))
        print('Loading Tiny Image indices from file {}'.format(ti_indices_filepath))
        assert pathlib.Path(ti_indices_filepath).is_file()
        with open(ti_indices_filepath, 'r') as f:
            tinyimage_indices = json.load(f)
        assert type(tinyimage_indices) is list
        assert len(tinyimage_indices) == labels.shape[0]
        return imagedata, labels, tinyimage_indices

In [15]:
images, labels = load_new_test_data('v6')

Loading labels from file /home/xueqiong/noise_injection_uncertainty/cifar10/CIFAR-10.1-master/datasets/cifar10.1_v6_labels.npy
Loading image data from file /home/xueqiong/noise_injection_uncertainty/cifar10/CIFAR-10.1-master/datasets/cifar10.1_v6_data.npy


In [16]:
labels = labels.astype('int')

In [17]:
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, data, targets, transform=None):
        self.data = data
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        x = self.data[index]
        y = self.targets[index]

        if self.transform:
            x = self.transform(x)

        return x, y

# # 创建自定义数据集实例
custom_dataset = CustomDataset(images, labels, test_transform)
ood_loader = torch.utils.data.DataLoader(custom_dataset, batch_size=200, shuffle=False,
                                            num_workers=args.workers, pin_memory=False)

In [18]:
# ood_dataset = dset.CIFAR100(
#         args.data_path, train=True, transform=train_transform, download=True)
# ood_loader = torch.utils.data.DataLoader(ood_dataset, batch_size=200, shuffle=False,
#                                             num_workers=args.workers, pin_memory=False)

In [19]:
def validate2(val_loader, model, criterion, log, test_times=10, return_detail=False):
    # switch to evaluate mode
    samplesize = val_loader.dataset.data.shape[0]
    model.eval()
    if 'dropout' in args.arch:
        recursive_module_iteration(model)
    with torch.no_grad():
        var_logits_all = []
        pred_all = []
        confidence_all  = []
        target_all = []
        avg_probs_all = []
        for i, (input, target) in enumerate(val_loader):
            logits = torch.zeros((input.shape[0], num_classes, test_times))
            if args.use_cuda:
                target = target.cuda(non_blocking=True)
                input = input.cuda()
                logits = logits.cuda()
            for i in range(test_times):
                # compute output
                output = model(input)
                logits[:, :, i] = output
            avg_logits = torch.mean(logits, dim=2)
            avg_probs = torch.nn.functional.softmax(avg_logits, dim=1)
            var_logits = torch.sum(torch.var(logits, dim=2), dim=-1)   # [batch_size]
            if args.use_cuda:
                var_logits = var_logits.cpu()
            var_logits = var_logits.numpy().tolist()
            _, pred = output.topk(1, 1, True, True)
            if args.use_cuda:
                pred = pred.cpu()
            pred = pred.t()[0].numpy().tolist()
            if return_detail:
                var_logits_all.extend(var_logits)
                pred_all.extend(pred)
                confidence, prediction = torch.max(avg_probs, 1)
                confidence_all.extend(confidence.cpu().numpy().tolist())
                avg_probs_all.extend(avg_probs.cpu().numpy().tolist())

    return None, var_logits_all, pred_all, confidence_all, target_all, None, avg_probs_all

cur_acc_ood, var_logits_all_ood, pred_all_ood, confidence_all_ood, target_all_ood, bs_ood, avg_probs_all_ood = validate(ood_loader, net, criterion, log, 100, return_detail=True)
# cur_acc_ood, var_logits_all_ood, pred_all_ood, confidence_all_ood, target_all_ood, bs_ood, avg_probs_all_ood = validate2(ood_loader, net, criterion, log, 100, return_detail=True)

  **Test** Prec@1 58.650


In [20]:
# v6
from sklearn.metrics import roc_auc_score

# print(1-roc_auc_score(true_or_false, var_logits_all+var_logits_all_ood))
# print(1-roc_auc_score(true_or_false[:10000], var_logits_all))
true_or_false = np.concatenate([np.asarray(pred_all)==np.asarray(target_all), np.asarray(pred_all_ood)==np.asarray(target_all_ood)])
print(roc_auc_score(true_or_false, confidence_all+confidence_all_ood))
print(roc_auc_score(true_or_false[:10000], confidence_all))

print(roc_auc_score(true_or_false[:10000], confidence_all))

ood_or_not = [0]*len(test_data)+[1]*len(ood_loader.dataset)
print(roc_auc_score(ood_or_not, confidence_all+confidence_all_ood))

0.8049533730364591
0.8133823210950126
0.8133823210950126
0.417994925


In [21]:
# # cifar100
# from sklearn.metrics import roc_auc_score

# # print(1-roc_auc_score(true_or_false, var_logits_all+var_logits_all_ood))
# # print(1-roc_auc_score(true_or_false[:10000], var_logits_all))
# ood_or_not = [0]*len(test_data)+[1]*len(ood_loader.dataset)
# print(roc_auc_score(ood_or_not, confidence_all+confidence_all_ood))

In [22]:
# np.array(avg_probs_all).shape

In [23]:
# # 将类别标签转换为one-hot编码
# one_hot_preds = np.zeros((len(pred_all), num_classes))
# one_hot_preds[np.arange(len(pred_all)), pred_all] = 1

In [24]:
# one_hot_preds

In [25]:
# from sklearn.metrics import mutual_info_score
# mis = [mutual_info_score(avg_probs_all[i], one_hot_preds[i]) for i in range(len(avg_probs_all))]